# 06. Simulaci?n de Producci?n por Lotes ? OralSolutions

Construcci?n de los **8 lotes cronol?gicos de producci?n** (`L01`?`L08`, 826 sesiones entre `2025-08-01` y `2025-11-01`) respetando el horizonte de **60 d?as** para la llegada del target, m?s dos lotes con deriva inducida (`L09_deriva_datos` y `L10_deriva_conceptual`), y auditor?a de las predicciones servidas por la API (`docs/predicciones.csv`).


In [1]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.pipeline import FECHA_CORTE, FECHA_EVENTO, TARGET, cargar_datos, construir_features
from monitoreo.simular_produccion import construir_lotes_simulacion

df_completo = construir_features(cargar_datos())
corte_ts = pd.Timestamp(FECHA_CORTE)
df_train = df_completo[df_completo[FECHA_EVENTO] < corte_ts].copy().reset_index(drop=True)
df_test = df_completo[df_completo[FECHA_EVENTO] >= corte_ts].copy().reset_index(drop=True)

print(f"OBT cargada: {len(df_completo):,} filas | Train: {len(df_train):,} | Test: {len(df_test):,}")


OBT cargada: 2,377 filas | Train: 1,551 | Test: 826


In [2]:
# Construir los 10 lotes (L01-L08 reales + L09 deriva de datos + L10 deriva conceptual)
lotes = construir_lotes_simulacion(df_test)

resumen_lotes = []
for lote_id, tipo_lote, df_lote in lotes:
    f_ini = pd.to_datetime(df_lote[FECHA_EVENTO]).min()
    f_fin = pd.to_datetime(df_lote[FECHA_EVENTO]).max()
    resumen_lotes.append({
        "lote_id": lote_id,
        "tipo_lote": tipo_lote,
        "n_sesiones": len(df_lote),
        "fecha_inicio": f_ini.strftime("%Y-%m-%d"),
        "fecha_fin": f_fin.strftime("%Y-%m-%d"),
        "target_disponible": (f_fin + pd.Timedelta(days=60)).strftime("%Y-%m-%d"),
        "tasa_abandono_real": round(float(df_lote[TARGET].mean()), 4),
    })

pd.DataFrame(resumen_lotes)


,lote_id,tipo_lote,n_sesiones,fecha_inicio,fecha_fin,target_disponible,tasa_abandono_real
0,L01,produccion_real,104,2025-08-01,2025-08-18,2025-10-17,0.6154
1,L02,produccion_real,104,2025-08-18,2025-08-30,2025-10-29,0.5769
2,L03,produccion_real,103,2025-08-30,2025-09-10,2025-11-09,0.5631
3,L04,produccion_real,103,2025-09-10,2025-09-22,2025-11-21,0.6214
4,L05,produccion_real,103,2025-09-22,2025-10-03,2025-12-02,0.6214
5,L06,produccion_real,103,2025-10-03,2025-10-13,2025-12-12,0.6311
6,L07,produccion_real,103,2025-10-13,2025-10-23,2025-12-22,0.6505
7,L08,produccion_real,103,2025-10-23,2025-11-01,2025-12-31,0.7087
8,L09_deriva_datos,deriva_datos_inducida,103,2025-11-06,2025-11-15,2026-01-14,0.7087
9,L10_deriva_conceptual,deriva_conceptual_inducida,103,2025-11-20,2025-11-29,2026-01-28,0.4660


### Auditor?a de trazabilidad de `docs/predicciones.csv`
Cada predicci?n registrada incluye `id_prediccion`, `timestamp_prediccion`, `entradas_json`, `probabilidad`, `decision`, `version_modelo`, `run_id` y `latencia_ms`.


In [3]:
df_pred = pd.read_csv(ROOT / "docs" / "predicciones.csv")
print(f"Total predicciones auditadas: {len(df_pred):,}")
print(f"Versiones de modelo servidas: {df_pred['version_modelo'].unique().tolist()}")
print(f"Latencia p50: {df_pred['latencia_ms'].median():.2f} ms | p95: {df_pred['latencia_ms'].quantile(0.95):.2f} ms")
df_pred.groupby("lote_id").agg(
    n_pred=("id_prediccion", "count"),
    prob_media=("probabilidad", "mean"),
    tasa_contactar=("decision", lambda s: (s == "contactar").mean()),
    latencia_p95_ms=("latencia_ms", lambda s: s.quantile(0.95)),
).round(4)


Total predicciones auditadas: 1,032
Versiones de modelo servidas: [7]
Latencia p50: 5.87 ms | p95: 6.61 ms


,n_pred,prob_media,tasa_contactar,latencia_p95_ms
lote_id,,,,
L01,104,0.6032,0.8558,6.6198
L02,104,0.6153,0.8846,6.6357
L03,103,0.5774,0.9417,6.4822
L04,103,0.6147,0.8835,6.7282
L05,103,0.6171,0.8641,6.4831
L06,103,0.5987,0.9029,6.4441
L07,103,0.6814,0.8932,6.3830
L08,103,0.6871,0.8835,6.5550
L09_deriva_datos,103,0.6828,0.7476,6.8809
